# Loan default prediction: 19-input training and evaluation

This notebook builds a **new, reproducible model** with the same input schema as
the existing FastAPI app. It does not recover the undocumented training process
of the old pickle, and the old notebook's ROC-AUC of 0.7387 does not apply here.

Run every cell in order. Outputs are written to `artifacts_19/`; your original
`loan_default_model.pkl` is never overwritten. This is an educational portfolio
project, not a validated lending decision system.

**Data:** LendingClub accepted loans, 2007-2018 Q4. Obtain
`accepted_2007_to_2018Q4.csv` separately and respect the source's redistribution
terms. Do not commit the raw data. The local copy is discovered below; change
`DATA_PATH` if needed. No data is downloaded by this notebook.

**Prediction question:** among issued, accepted loans with observed final
outcomes, distinguish `Charged Off` from `Fully Paid`. Interest rate and grade
are known at issuance; this is not a pre-underwriting approval model. Current,
late, missing, and other statuses are excluded, not treated as repaid.

**Important limitation:** resolved outcomes in this historical snapshot are a
selected subset. Newer loans have less follow-up. Chronological separation does
not eliminate this maturity bias or establish a prospective fixed-horizon test.
Training uses final outcomes from the snapshot; it is not a reconstruction of
what labels were available on each historical cutoff date.


In [1]:
from pathlib import Path
import sys, json, hashlib, platform, importlib.metadata, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay, brier_score_loss,
)

SEED = 42
THREADS = 4
candidates = [
    Path('data/raw/accepted_2007_to_2018Q4.csv'),
    Path('accepted_2007_to_2018Q4.csv'),
]
DATA_PATH = next((p for p in candidates if p.is_file()), candidates[0])
assert DATA_PATH.is_file(), f'Set DATA_PATH to your CSV. Tried: {candidates}'
ARTIFACT_DIR = Path('artifacts_19')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length',
    'annual_inc', 'pymnt_plan', 'dti', 'delinq_2yrs', 'fico_range_low',
    'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'credit_history_years',
]
assert len(FEATURES) == 19 and len(set(FEATURES)) == 19
versions = {name: importlib.metadata.version(name) for name in
            ['numpy', 'pandas', 'scikit-learn', 'lightgbm', 'joblib', 'matplotlib']}
print('Python:', platform.python_version())
print('Packages:', versions)
print('Dataset:', DATA_PATH.resolve())


Python: 3.12.14
Packages: {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'lightgbm': '4.6.0', 'joblib': '1.5.2', 'matplotlib': '3.10.7'}
Dataset: C:\Loan-Default-Risk-App\accepted_2007_to_2018Q4.csv


## 1. Load only necessary columns and define the target

Row-level conversions below are deterministic. Imputation, scaling, model
training, and threshold selection happen later using only the appropriate split.

`pymnt_plan` may describe payment activity after origination. To avoid relying
on that information, it is **forced to zero**. It remains a non-predictive
placeholder so the model accepts the existing 19-field API schema. Thus this
version has 19 input fields and at most 18 informative predictors. Do not claim
all 19 were demonstrated to be the most important features. Other variables
also need a source data dictionary review before any real deployment.


In [2]:
raw_columns = [f for f in FEATURES if f != 'credit_history_years'] + [
    'loan_status', 'issue_d', 'earliest_cr_line', 'id',
]
raw = pd.read_csv(DATA_PATH, usecols=raw_columns, low_memory=False)
raw_rows = len(raw)
status_counts = raw['loan_status'].value_counts(dropna=False).astype(int)
print('Raw rows:', f'{raw_rows:,}')
print('Status counts:\n', status_counts.to_string())
data = raw.loc[raw['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
del raw
before = len(data)
# Deduplicate by loan ID, not by shared borrower characteristics.
identified = data['id'].notna()
data = data.loc[~(identified & data['id'].duplicated())].copy()
duplicate_ids_removed = before - len(data)
data['target'] = data['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1}).astype('int8')
data['issue_date'] = pd.to_datetime(data['issue_d'], format='%b-%Y', errors='coerce')
earliest = pd.to_datetime(data['earliest_cr_line'], format='%b-%Y', errors='coerce')
# Match the original app project's whole-calendar-year definition.
data['credit_history_years'] = data['issue_date'].dt.year - earliest.dt.year
data.loc[data['credit_history_years'] < 0, 'credit_history_years'] = np.nan
data['term'] = pd.to_numeric(data['term'].astype('string').str.extract(r'(\d+)', expand=False), errors='coerce')
data['grade'] = data['grade'].map({letter: i for i, letter in enumerate('ABCDEFG')})
employment = {'< 1 year': 0, '1 year': 1, '10+ years': 10}
employment.update({f'{i} years': i for i in range(2, 10)})
data['emp_length'] = data['emp_length'].map(employment)
original_payment_plan_counts = data['pymnt_plan'].value_counts(dropna=False).astype(int)
data['pymnt_plan'] = 0
for column in FEATURES:
    data[column] = pd.to_numeric(data[column], errors='coerce')
data[FEATURES] = data[FEATURES].replace([np.inf, -np.inf], np.nan)
# Treat invalid values as missing, using fixed rules rather than learned cutoffs.
for column in FEATURES:
    data.loc[data[column] < 0, column] = np.nan
for column in ['fico_range_low', 'fico_range_high']:
    data.loc[~data[column].between(300, 850), column] = np.nan
data.loc[~data['term'].isin([36, 60]), 'term'] = np.nan
data.loc[~data['revol_util'].between(0, 100), 'revol_util'] = np.nan
date_missing_removed = int(data['issue_date'].isna().sum())
data = data.dropna(subset=['issue_date']).copy()
data = data[FEATURES + ['target', 'issue_date']].sort_values('issue_date').reset_index(drop=True)
print('Eligible rows:', f'{len(data):,}', '| duplicate IDs removed:', duplicate_ids_removed)
print('Missing issue dates removed:', date_missing_removed)
print('Default prevalence:', round(float(data['target'].mean()), 4))
print('Missing feature values (before training-only imputation):')
print(data[FEATURES].isna().sum().to_string())


Raw rows: 2,260,701
Status counts:
 loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Eligible rows: 1,345,310 | duplicate IDs removed: 0
Missing issue dates removed: 0
Default prevalence: 0.1996
Missing feature values (before training-only imputation):
loan_amnt                   0
term                        0
int_rate                    0
installment                 0
grade                       0
emp_

## 2. Separate train, validation, and final test data chronologically

The first approximately 70% of eligible rows are training data, the next 15%
validation, and the latest 15% test. Cutoffs use issuance dates without looking
at outcomes; entire issue months stay together. Final proportions vary because
months have different numbers of loans. Validation selects boosting iterations
and a classification threshold. The test set is evaluated only after these
choices are fixed. Do not repeatedly modify this model based on test results.


In [3]:
train_end = data['issue_date'].iloc[int(len(data) * 0.70)]
validation_end = data['issue_date'].iloc[int(len(data) * 0.85)]
train = data.loc[data['issue_date'] < train_end]
validation = data.loc[(data['issue_date'] >= train_end) & (data['issue_date'] < validation_end)]
test = data.loc[data['issue_date'] >= validation_end]
X_train, y_train = train[FEATURES], train['target']
X_val, y_val = validation[FEATURES], validation['target']
X_test, y_test = test[FEATURES], test['target']
split_summary = {}
for name, frame in [('train', train), ('validation', validation), ('test', test)]:
    assert len(frame) > 0 and frame['target'].nunique() == 2, f'Invalid {name} split'
    split_summary[name] = {
        'rows': len(frame), 'default_rate': float(frame['target'].mean()),
        'start': str(frame['issue_date'].min().date()),
        'end': str(frame['issue_date'].max().date()),
    }
assert train['issue_date'].max() < validation['issue_date'].min()
assert validation['issue_date'].max() < test['issue_date'].min()
print(json.dumps(split_summary, indent=2))


{
  "train": {
    "rows": 938537,
    "default_rate": 0.18655844148925402,
    "start": "2007-06-01",
    "end": "2016-03-01"
  },
  "validation": {
    "rows": 197556,
    "default_rate": 0.248896515418413,
    "start": "2016-04-01",
    "end": "2017-01-01"
  },
  "test": {
    "rows": 209217,
    "default_rate": 0.2117227567549482,
    "start": "2017-02-01",
    "end": "2018-12-01"
  }
}


## 3. Fit preprocessing and baseline models on training data only

The dummy baseline predicts the training default prevalence for every loan.
Logistic regression is a stronger comparison. Both are fitted on training data,
without class reweighting. This also avoids treating a class-weighted model's
raw scores as calibrated default probabilities.


In [4]:
imputer = SimpleImputer(strategy='median', keep_empty_features=True).set_output(transform='pandas')
X_train_ready = imputer.fit_transform(X_train)
X_val_ready = imputer.transform(X_val)
assert X_train_ready.shape[1] == 19 and not X_train_ready.isna().any().any()
dummy = DummyClassifier(strategy='prior', random_state=SEED).fit(X_train_ready, y_train)
logistic = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=SEED)),
]).fit(X_train_ready, y_train)
print('Training-only imputation values:')
print(dict(zip(FEATURES, imputer.statistics_)))


Training-only imputation values:
{'loan_amnt': np.float64(12075.0), 'term': np.float64(36.0), 'int_rate': np.float64(12.69), 'installment': np.float64(379.37), 'grade': np.float64(2.0), 'emp_length': np.float64(6.0), 'annual_inc': np.float64(65000.0), 'pymnt_plan': np.float64(0.0), 'dti': np.float64(17.61), 'delinq_2yrs': np.float64(0.0), 'fico_range_low': np.float64(690.0), 'fico_range_high': np.float64(694.0), 'inq_last_6mths': np.float64(0.0), 'open_acc': np.float64(11.0), 'pub_rec': np.float64(0.0), 'revol_bal': np.float64(11607.0), 'revol_util': np.float64(54.9), 'total_acc': np.float64(24.0), 'credit_history_years': np.float64(15.0)}


## 4. Fit LightGBM using validation data for early stopping

Fixed starting settings: up to 500 trees, learning rate 0.05, 31 leaves,
random seed 42. Stop when validation ROC-AUC fails to improve for 50 rounds.
There is no test data in `eval_set`. Feature selection and Optuna tuning are
not claimed or performed in this notebook.


In [5]:
model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    random_state=SEED, n_jobs=THREADS, verbosity=-1, deterministic=True,
    force_col_wise=True,
)
model.fit(
    X_train_ready, y_train,
    eval_set=[(X_val_ready, y_val)], eval_metric='auc',
    callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False)],
)
assert model.n_features_in_ == 19
print('Selected iteration:', model.best_iteration_)


Selected iteration: 500


## 5. Choose thresholds on validation data, then evaluate the test set

For each nonconstant model, choose the threshold maximizing validation F1.
This is a demonstration objective, not a lending cost function. A real policy
would require a justified false-positive/false-negative cost and additional
validation. For the constant dummy model, keep threshold 0.5.

ROC-AUC and average precision use continuous scores. Precision, recall, F1,
accuracy, and the confusion matrix use the frozen classification threshold.
Report the prevalence and average precision together when discussing imbalance.
Model outputs are estimates; this notebook does not establish calibration.


In [6]:
models = {'dummy': dummy, 'logistic_regression': logistic, 'lightgbm': model}
thresholds = {}
for name, estimator in models.items():
    scores = estimator.predict_proba(X_val_ready)[:, 1]
    if name == 'dummy':
        thresholds[name] = 0.5
    else:
        precision, recall, candidates = precision_recall_curve(y_val, scores)
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        thresholds[name] = float(candidates[int(np.argmax(f1))])
print('Frozen validation-selected thresholds:', thresholds)

X_test_ready = imputer.transform(X_test)
results, test_scores = {}, {}
for name, estimator in models.items():
    scores = estimator.predict_proba(X_test_ready)[:, 1]
    predictions = (scores >= thresholds[name]).astype(int)
    test_scores[name] = scores
    results[name] = {
        'roc_auc': float(roc_auc_score(y_test, scores)),
        'average_precision': float(average_precision_score(y_test, scores)),
        'precision': float(precision_score(y_test, predictions, zero_division=0)),
        'recall': float(recall_score(y_test, predictions, zero_division=0)),
        'f1': float(f1_score(y_test, predictions, zero_division=0)),
        'accuracy': float(accuracy_score(y_test, predictions)),
        'brier_score': float(brier_score_loss(y_test, scores)),
        'threshold': thresholds[name],
        'confusion_matrix': confusion_matrix(y_test, predictions, labels=[0, 1]).tolist(),
    }
comparison = pd.DataFrame(results).T
print('FINAL TEST RESULTS (new model; not the old pickle):')
print(comparison.drop(columns='confusion_matrix').round(4).to_string())
print('\nLightGBM classification report:')
print(classification_report(y_test, (test_scores['lightgbm'] >= thresholds['lightgbm']).astype(int),
                            target_names=['Fully Paid', 'Charged Off'], zero_division=0))
print('Confusion matrix: rows = actual, columns = predicted; class order [0, 1]')
print(results['lightgbm']['confusion_matrix'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name in ['logistic_regression', 'lightgbm']:
    RocCurveDisplay.from_predictions(y_test, test_scores[name], name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, test_scores[name], name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[1].axhline(float(y_test.mean()), linestyle='--', color='gray', label='Test prevalence')
axes[1].legend()
ConfusionMatrixDisplay(np.array(results['lightgbm']['confusion_matrix']),
                       display_labels=['Paid', 'Charged Off']).plot(ax=axes[2], colorbar=False)
fig.suptitle('19-input loan model: chronological held-out test')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'evaluation.png', dpi=160)
plt.close(fig)


Frozen validation-selected thresholds: {'dummy': 0.5, 'logistic_regression': 0.14828826514086613, 'lightgbm': 0.1678487251788922}
FINAL TEST RESULTS (new model; not the old pickle):
                      roc_auc average_precision precision    recall        f1  accuracy brier_score threshold
dummy                     0.5          0.211723       0.0       0.0       0.0  0.788277    0.167529       0.5
logistic_regression  0.687843          0.350749  0.295764  0.746433  0.423658  0.570016    0.156961  0.148288
lightgbm             0.701159           0.36702  0.316261  0.694081  0.434528  0.617526    0.154593  0.167849

LightGBM classification report:
              precision    recall  f1-score   support

  Fully Paid       0.88      0.60      0.71    164921
 Charged Off       0.32      0.69      0.43     44296

    accuracy                           0.62    209217
   macro avg       0.60      0.65      0.57    209217
weighted avg       0.76      0.62      0.65    209217

Confusion matrix: 

## 6. Save the evaluated pipeline, metadata, and an API example

The saved pipeline includes the fitted imputer and model. It accepts a numeric
DataFrame with these exact 19 columns in order. Its `.predict()` uses 0.5;
**the reported classification results instead use the validation-selected
threshold in `metrics_19.json`**. Apply that threshold explicitly in the API.

The saved example uses training-set medians, not a real borrower's record.
Numeric encodings: grade A-G = 0-6; employment `< 1 year` = 0, `10+ years` = 10;
rate and revolving utilization use percentage points (13.99 means 13.99%);
term uses months; credit history uses difference in calendar years;
payment-plan placeholder is 0. Unknown employment is missing, not an eleventh
employment-duration category. This differs from some earlier experiments.


In [7]:
pipeline = Pipeline([('imputer', imputer), ('model', model)])
model_path = ARTIFACT_DIR / 'loan_default_19_pipeline.joblib'
joblib.dump(pipeline, model_path)
reloaded = joblib.load(model_path)
sample = X_test.iloc[:100].copy()
np.testing.assert_allclose(reloaded.predict_proba(sample), pipeline.predict_proba(sample), rtol=0, atol=1e-12)
assert list(reloaded.feature_names_in_) == FEATURES
assert reloaded.n_features_in_ == 19

metadata = {
    'model_description': 'Newly trained 19-input pipeline; not a reconstruction of the old model',
    'seed': SEED, 'python_version': platform.python_version(), 'package_versions': versions,
    'dataset_filename': DATA_PATH.name, 'dataset_size_bytes': DATA_PATH.stat().st_size,
    'raw_rows': raw_rows, 'eligible_rows': len(data),
    'duplicate_ids_removed': duplicate_ids_removed,
    'missing_issue_dates_removed': date_missing_removed,
    'target': {'0': 'Fully Paid', '1': 'Charged Off'},
    'features': FEATURES, 'split': split_summary,
    'pymnt_plan_policy': 'Forced to zero; retained only for schema compatibility',
    'best_iteration': int(model.best_iteration_),
    'decision_rule': 'default_probability >= lightgbm.threshold',
    'test_results': results,
    'limitations': [
        'Only accepted loans with resolved outcomes in the snapshot are evaluated.',
        'Newer cohorts have shorter follow-up; maturity and selection bias remain.',
        'Chronological cohorts do not simulate historical label availability.',
        'Default probability calibration and subgroup fairness are not established.',
        'Features were chosen to match the existing API, not proven optimal.',
    ],
}
(ARTIFACT_DIR / 'metrics_19.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
comparison.drop(columns='confusion_matrix').to_csv(ARTIFACT_DIR / 'model_comparison.csv', index_label='model')
(ARTIFACT_DIR / 'feature_columns_19.json').write_text(json.dumps(FEATURES, indent=2), encoding='utf-8')
example = {feature: float(value) for feature, value in zip(FEATURES, imputer.statistics_)}
integer_fields = ['term', 'grade', 'emp_length', 'pymnt_plan', 'credit_history_years']
for feature in integer_fields:
    example[feature] = int(example[feature])
example_frame = pd.DataFrame([example], columns=FEATURES)
example_probability = float(reloaded.predict_proba(example_frame)[0, 1])
example_output = {
    'default_probability': example_probability,
    'prediction': 'Default' if example_probability >= thresholds['lightgbm'] else 'Fully Paid',
    'threshold': thresholds['lightgbm'],
}
(ARTIFACT_DIR / 'example_request.json').write_text(json.dumps(example, indent=2), encoding='utf-8')
(ARTIFACT_DIR / 'example_response.json').write_text(json.dumps(example_output, indent=2), encoding='utf-8')
(ARTIFACT_DIR / 'verified_requirements.txt').write_text(
    '\n'.join(f'{name}=={version}' for name, version in versions.items()) + '\n', encoding='utf-8')
print('Reloaded pipeline verified against 100 held-out examples.')
print('Artifacts:', ARTIFACT_DIR.resolve())
print('Example output:', example_output)


Reloaded pipeline verified against 100 held-out examples.
Artifacts: C:\Users\alahd\Documents\Codex\2026-11-05\hi\outputs\artifacts_19
Example output: {'default_probability': 0.16185353724903404, 'prediction': 'Fully Paid', 'threshold': 0.1678487251788922}


## 7. Run the connected API and prepare GitHub

The repository's `app/main.py` now loads `artifacts_19/loan_default_19_pipeline.joblib`,
reads the exact feature order from `feature_columns_19.json`, and uses the threshold
from `metrics_19.json` for both prediction and risk labels. The pipeline's default
`.predict()` threshold is 0.5; the API deliberately uses `.predict_proba()` and the
recorded validation-selected threshold instead.

From the repository root, install `requirements.txt` and start:

```powershell
python -m uvicorn app.main:app --reload
```

Visit `http://127.0.0.1:8000/docs`. For tests, install `requirements-dev.txt` and
run `python -m pytest -q`. The tests check the saved example, startup, input
validation, imputation, and the threshold. Model serialization is checked by this
notebook; API integration is checked separately by the tests.

The API accepts explicit nulls for nullable fields and uses the saved imputer.
Input encodings and units are documented in the API schema and README. The
payment-plan placeholder must be 0. The API adds `risk_level` to the core fields
recorded in `example_response.json`.

Upload the notebook, app, tests, requirements, README, results, and `artifacts_19`.
Do not upload `data/`, cache files, virtual environments, or old reference folders.
The included `.gitignore` excludes these. Raw data download instructions are in
the README. Credit collaborators and describe your own contribution accurately.

References: [scikit-learn: avoiding leakage](https://scikit-learn.org/stable/common_pitfalls.html),
[LightGBM: early stopping](https://lightgbm.readthedocs.io/en/stable/Python-Intro.html),
[FastAPI: testing lifespan](https://fastapi.tiangolo.com/advanced/testing-events/).
